# 🔴 Vision Studio Engine
**Dedicated High-Fidelity Image Generation Environment**

This notebook is optimized for production. It utilizes Google Drive caching to eliminate redundant model downloads and features an enterprise-grade Gradio interface with VRAM management and versatile bulk rendering capabilities.

In [ ]:
#@title 📁 Step 1 — Mount Drive & Install Dependencies
import os
from google.colab import drive

# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Define the folder path in your Drive where the model will live
CACHE_DIR = '/content/drive/MyDrive/HF_Model_Cache'
os.makedirs(CACHE_DIR, exist_ok=True)

# 3. Tell Hugging Face to use this Drive folder
os.environ['HF_HOME'] = CACHE_DIR
print(f"✅ Hugging Face cache securely routed to: {CACHE_DIR}")

# 4. Install required packages
!pip install -q --upgrade pip
!pip install -q \
    'transformers>=4.51.0' \
    'diffusers>=0.32.0' \
    'accelerate>=0.30.0' \
    'gradio>=4.44.0' \
    'safetensors'

print('✅ All dependencies installed.')

In [ ]:
#@title 🔑 Step 2 — HuggingFace Login (Optional)
from huggingface_hub import login
login()  # Paste your HF token here if you are using private/gated models

In [ ]:
#@title 🧠 Step 3 — Load Image Engine & Memory Manager
import torch, gc
from diffusers import StableDiffusionXLPipeline
import gradio as gr
import random

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  System Architecture: {DEVICE.upper()}')
if DEVICE == 'cuda':
    print(f'    Hardware Accelerator : {torch.cuda.get_device_name(0)}')

# Global state memory
loaded_model = None

IMG_MODEL_ID = 'John6666/janku-v5-nsfw-trained-noobai-rou-wei-illustrious-xl-v50-sdxl'

def load_image_model():
    global loaded_model
    if loaded_model is not None:
        return "**System Status:** 🟢 Engine Online"
    
    print(f'⏳ Initializing weights from: {IMG_MODEL_ID}')
    print('   (Loading from Google Drive cache if available...)')
    
    pipe = StableDiffusionXLPipeline.from_pretrained(
        IMG_MODEL_ID,
        torch_dtype=torch.float16,
        use_safetensors=True
    )
    pipe = pipe.to(DEVICE)
    pipe.enable_attention_slicing()
    pipe.enable_vae_slicing()
    pipe.enable_vae_tiling()
    
    loaded_model = pipe
    print('✅ Neural engine loaded into VRAM successfully.')
    return "**System Status:** 🟢 Engine Online"

def purge_vram():
    global loaded_model
    if loaded_model is not None:
        del loaded_model
        loaded_model = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print('🗑️ Critical: VRAM purged successfully.')
    return "**System Status:** 🔴 Offline (Memory Purged)"

def generate_image(prompt, negative_prompt, gen_mode, batch_size, steps, cfg, width, height, seed):
    global loaded_model
    if loaded_model is None:
        raise gr.Error("⚠️ Critical Error: Please initialize the engine before requesting a render.")
        
    generated_images = []
    base_seed = int(seed) if seed >= 0 else random.randint(0, 2147483647)
    full_neg = f'worst quality, low quality, normal quality, jpeg artifacts, blurry, {negative_prompt}'

    if gen_mode == 'Different Prompts (Line-by-Line)':
        # Split the prompt box by newlines, ignoring empty lines
        prompts_list = [p.strip() for p in prompt.split('\n') if p.strip()]
        if not prompts_list:
            raise gr.Error("⚠️ Critical Error: Please provide at least one prompt.")
            
        for i, p in enumerate(prompts_list):
            current_seed = base_seed + i
            generator = torch.Generator(device=DEVICE).manual_seed(current_seed)
            full_prompt = f'masterpiece, best quality, newest, absurdres, {p}'

            with torch.autocast(DEVICE, dtype=torch.float16):
                result = loaded_model(
                    prompt=full_prompt,
                    negative_prompt=full_neg,
                    num_inference_steps=int(steps),
                    guidance_scale=float(cfg),
                    width=int(width),
                    height=int(height),
                    generator=generator
                )
            generated_images.append((result.images[0], f"Prompt {i+1} | Seed: {current_seed}"))
            
    else:
        # Same Prompt Mode
        full_prompt = f'masterpiece, best quality, newest, absurdres, {prompt}'
        for i in range(int(batch_size)):
            current_seed = base_seed + i
            generator = torch.Generator(device=DEVICE).manual_seed(current_seed)

            with torch.autocast(DEVICE, dtype=torch.float16):
                result = loaded_model(
                    prompt=full_prompt,
                    negative_prompt=full_neg,
                    num_inference_steps=int(steps),
                    guidance_scale=float(cfg),
                    width=int(width),
                    height=int(height),
                    generator=generator
                )
            generated_images.append((result.images[0], f"Seed: {current_seed}"))

    return generated_images

print('✅ Core architecture ready.')

In [ ]:
#@title 🎨 Step 4 — Launch Studio Interface
import gradio as gr

# Enterprise Red & Zinc Theme
theme = gr.themes.Default(
    primary_hue='red',
    secondary_hue='rose',
    neutral_hue='zinc',
    font=[gr.themes.GoogleFont('Inter'), 'ui-sans-serif', 'system-ui', 'sans-serif']
).set(
    button_primary_background_fill='*primary_600',
    button_primary_background_fill_hover='*primary_700',
    button_primary_text_color='white',
    block_title_text_weight='600',
    block_border_width='1px',
    block_shadow='*shadow_sm',
    slider_color='*primary_600',
    accordion_text_color='*neutral_800'
)

with gr.Blocks(theme=theme, title='Vision Studio Engine') as demo:

    with gr.Row():
        gr.Markdown("""
        # 🔴 Vision Studio Engine
        **NoobAI SDXL Implementation** · Optimized for Production
        """)

    with gr.Row(variant='panel'):
        with gr.Column(scale=1):
            img_load_btn = gr.Button('⚡ Initialize Engine', variant='secondary')
            vram_btn = gr.Button('🗑️ Purge VRAM', variant='stop')
        with gr.Column(scale=4):
            img_status = gr.Markdown('**System Status:** 🔴 Offline (Awaiting Initialization)')
            
    img_load_btn.click(fn=load_image_model, outputs=img_status)
    vram_btn.click(fn=purge_vram, outputs=img_status)
    
    gr.Markdown('<br>')

    with gr.Row():
        with gr.Column(scale=4):
            img_mode = gr.Radio(
                choices=['Same Prompt (Use Batch Size)', 'Different Prompts (Line-by-Line)'],
                value='Same Prompt (Use Batch Size)',
                label='Generation Mode',
                info="Select how you want to handle multiple images."
            )
            
            img_prompt = gr.Textbox(
                label='Subject Prompt(s)',
                placeholder='Describe the subject in detail...\n(If using Line-by-Line mode, press Enter for a new prompt!)',
                lines=5
            )
            img_neg = gr.Textbox(
                label='Negative Parameters',
                value='bad anatomy, extra limbs, watermark, censored, lowres',
                lines=2
            )
            
            with gr.Accordion('⚙️ Rendering Parameters', open=False):
                with gr.Row():
                    img_batch = gr.Slider(1, 10, value=1, step=1, label='Batch Size (Ignored in Line-by-Line mode)')
                    img_steps = gr.Slider(10, 60, value=28, step=1, label='Inference Steps')
                with gr.Row():
                    img_cfg = gr.Slider(1.0, 15.0, value=7.0, step=0.5, label='CFG Scale')
                    img_seed = gr.Number(value=-1, label='Seed (-1 for randomized)', precision=0)
                with gr.Row():
                    img_w = gr.Dropdown([512, 768, 832, 1024, 1216], value=832, label='Resolution (Width)')
                    img_h = gr.Dropdown([512, 768, 832, 1024, 1216], value=1216, label='Resolution (Height)')

            img_btn = gr.Button('Execute Render', variant='primary', size='lg')
                    
        with gr.Column(scale=5):
            img_out = gr.Gallery(label='Output Render(s)', columns=[1, 2], object_fit='contain', height='auto', interactive=False)
            
    img_btn.click(
        fn=generate_image,
        inputs=[img_prompt, img_neg, img_mode, img_batch, img_steps, img_cfg, img_w, img_h, img_seed],
        outputs=img_out
    )

demo.launch(
    share=True,
    debug=False,
    show_error=True
)
print('🚀 UI deployed successfully.')